# RubricJudge — Standalone Training, Optimization & Eval

Self-contained notebook for the RubricJudge module.

**Evaluates MCQ quality across 11 rubric criteria + overall_decision**  
**Input:** `questions: list[RubricQuestion]` (includes `language_variant`)  
**Output:** `output: RubricOutput` → `output.results: list[RubricResult]`  
**Datasets:** `training_dataset_standard.json` (training) · `eval_dataset_standard.json` (Promptfoo eval)  
**Flow:** Setup → Define Models → Load Data → Baseline Eval → GEPA Optimize → Post-Eval → Promptfoo

Run all cells top-to-bottom. No changes are made to any existing `.py` files.

## Cell 1 — Setup & Imports

In [1]:
import sys, os, json, subprocess, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# ── Locate project root (directory that contains utils.py) ──────────────────
_candidate = Path().resolve()
if (_candidate / 'utils.py').exists():
    PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists():
    PROJECT_ROOT = _candidate.parent
else:
    raise RuntimeError(
        f'Cannot locate project root from {_candidate}. '
        'Open Jupyter from d:/Topin or d:/Topin/notebooks.'
    )

NOTEBOOK_DIR  = PROJECT_ROOT / 'notebooks'
DATA_DIR      = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
EVALS_DIR     = NOTEBOOK_DIR / 'evals'

ARTIFACTS_DIR.mkdir(exist_ok=True)
EVALS_DIR.mkdir(parents=True, exist_ok=True)

OPTIMIZED_PATH = ARTIFACTS_DIR / 'rubric_agent_optimized.json'
EVAL_OUTPUT    = EVALS_DIR / 'rubric_eval_output.json'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Auto-inject venv site-packages so dspy is importable from any kernel ────
_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'      # Windows
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'  # Linux/macOS
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')
else:
    print(f'Venv site-packages path: {_venv_sp}  (exists={_venv_sp.exists()})')

import dspy

print(f'Project root : {PROJECT_ROOT}')
print(f'Artifacts    : {ARTIFACTS_DIR}')
print(f'Evals dir    : {EVALS_DIR}')
print(f'DSPy version : {dspy.__version__}')

Injected venv site-packages: D:\Topin\.venv\Lib\site-packages
Project root : D:\Topin
Artifacts    : D:\Topin\artifacts
Evals dir    : D:\Topin\notebooks\evals
DSPy version : 3.1.3


## Cell 2 — Configure DSPy (Mistral)

In [2]:
from utils import configure_dspy_from_env

lm = configure_dspy_from_env()
print(f'Active LM : {lm}')

Active LM : <dspy.clients.lm.LM object at 0x0000025F8B642230>


## Cell 3 — Define Models + Signature + Agent

- `RubricQuestion` — typed input model (MCQ question + `language_variant`)
- `RubricResult` — typed output model (11 rubric criteria + `overall_decision` + `priority_reason` + `revision_feedback`)
- `RubricOutput` — wrapper class holding `results: list[RubricResult]`
- `RubricJudgeSignature` — `questions: list[RubricQuestion]` → `output: RubricOutput`

**`overall_decision` values:** `Pass` · `Revise` · `Fail`  
**Ambiguity rule:** if `ambiguity == "Major Issue"` the decision must be `Fail`

In [3]:
from pydantic import BaseModel
from typing import Literal


# ── Input model ──────────────────────────────────────────────────────────────
class RubricQuestion(BaseModel):
    question_id:      str
    stem:             str
    options:          list[str]
    correct_answer:   str
    explanation:      str
    target_cefr:      str
    target_difficulty: str
    language_variant: str   # e.g. 'British English' or 'American English'


# ── Output models ─────────────────────────────────────────────────────────────
class RubricResult(BaseModel):
    question_id:                          str
    grammatical_accuracy:                 Literal['No Issues', 'Minor Issues', 'Major Issues']
    spelling:                             Literal['No Issues', 'Minor Issues', 'Major Issues']
    ambiguity:                            Literal['No Issue', 'Minor Issue', 'Major Issue']
    functionality_alignment:              Literal['Aligned', 'Partially Aligned', 'Not Aligned']
    instruction_clarity_appropriateness:  Literal['Clear', 'Needs Improvement', 'Unclear']
    academic_language_exam_acceptability: Literal['Acceptable', 'Needs Improvement', 'Not Acceptable']
    option_explanation_consistency:       Literal['Consistent', 'Inconsistent']
    readability:                          Literal['Good', 'Needs Improvement', 'Poor']
    formatting_spacing:                   Literal['No Issues', 'Minor Issues', 'Major Issues']
    punctuation:                          Literal['No Issues', 'Minor Issues', 'Major Issues']
    british_american_english_consistency: Literal['Consistent', 'Inconsistent']
    overall_decision:                     Literal['Pass', 'Revise', 'Fail']
    priority_reason:                      str
    revision_feedback:                    str


class RubricOutput(BaseModel):
    results: list[RubricResult]


# ── Signature ────────────────────────────────────────────────────────────────
class RubricJudgeSignature(dspy.Signature):
    """Evaluate a list of MCQ questions using the rubric.

    For EACH question, assess all 11 rubric criteria:
      1.  grammatical_accuracy                 — grammar correctness
      2.  spelling                             — spelling errors
      3.  ambiguity                            — HIGHEST PRIORITY: if Major Issue → must Fail
      4.  functionality_alignment              — does it test what it claims?
      5.  instruction_clarity_appropriateness  — instructions suitable for target CEFR?
      6.  academic_language_exam_acceptability — suitable for a formal exam?
      7.  option_explanation_consistency       — explanation matches correct option?
      8.  readability                          — readable at the target CEFR level?
      9.  formatting_spacing                   — formatting/spacing issues?
      10. punctuation                          — punctuation errors?
      11. british_american_english_consistency — consistent English variant?

    Decision rules:
      - ambiguity == 'Major Issue'  → overall_decision must be 'Fail'
      - Any Major Issue in any criterion → 'Fail' or 'Revise'
      - Minor issues only → 'Revise'
      - All criteria clean → 'Pass'

    Return one RubricResult per question in the same order as the input list.
    """
    questions: list[RubricQuestion] = dspy.InputField(desc='List of MCQ questions to be evaluated by the rubric')
    output:    RubricOutput         = dspy.OutputField(desc='Rubric evaluation results for all questions')


# ── Agent ─────────────────────────────────────────────────────────────────────
class RubricJudgeAgent(dspy.Module):
    """Batch rubric judge. Attribute `judge` serialises as `judge.predict` in the artifact."""

    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(RubricJudgeSignature)

    def forward(self, questions: list[RubricQuestion]) -> dspy.Prediction:
        return self.judge(questions=questions)


print('Models and agent defined.')
print('  Input  : questions: list[RubricQuestion]  (includes language_variant)')
print('  Output : output: RubricOutput')
print('           output.results → list[RubricResult]')
print('           key field: overall_decision (Pass / Revise / Fail)')

Models and agent defined.
  Input  : questions: list[RubricQuestion]  (includes language_variant)
  Output : output: RubricOutput
           output.results → list[RubricResult]
           key field: overall_decision (Pass / Revise / Fail)


## Cell 4 — Define Metric

Scores `overall_decision`: `Pass`=1.0 · `Revise`=0.5 · `Fail`=0.0  
Returns `(float, str)` tuple for GEPA — plain `float` for BootstrapFewShot.

In [4]:
DECISION_SCORE = {'Pass': 1.0, 'Revise': 0.5, 'Fail': 0.0}


def rubric_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Reads pred.output (RubricOutput) → pred.output.results[0] (RubricResult).
    GEPA: 5 args → returns (float, str).
    BootstrapFewShot: 3 args → returns plain float.
    """
    try:
        result   = pred.output.results[0]
        decision = result.overall_decision
    except Exception:
        msg = 'No valid output from prediction.'
        return (0.0, msg) if pred_name is not None else 0.0

    expected = gold.expected_overall_decision   # 'Pass', 'Revise', or 'Fail'
    score    = DECISION_SCORE.get(decision, 0.0)

    # Full credit only when prediction matches expected decision exactly
    if decision == expected:
        score = 1.0
    elif (decision == 'Revise' and expected == 'Pass') or \
         (decision == 'Pass'   and expected == 'Revise'):
        score = 0.5   # close but not exact
    else:
        score = 0.0   # completely wrong

    feedback = (
        f'Expected overall_decision={expected} (got "{decision}"). '
        f'Score: {score:.1f}. '
        f'priority_reason: {result.priority_reason}. '
        + ('Correct.' if decision == expected
           else f'Focus on: ambiguity (highest priority), then grammatical_accuracy, '
                f'functionality_alignment, and option_explanation_consistency.')
    )

    return (score, feedback) if pred_name is not None else score


print('rubric_metric defined  (reads pred.output.results[0].overall_decision).')
print('  Pass=1.0  |  Revise=0.5  |  Fail=0.0  (exact match scoring)')

rubric_metric defined  (reads pred.output.results[0].overall_decision).
  Pass=1.0  |  Revise=0.5  |  Fail=0.0  (exact match scoring)


## Cell 5 — Load Datasets

| File | Role | Questions |
|------|------|-----------|
| `data/training_dataset_standard.json` | Optimizer training | 24 (4 per CEFR A1–C2) |
| `data/eval_dataset_standard.json` | Promptfoo eval only | 24 (4 per CEFR A1–C2) |

`language_variant` defaults to `'British English'` if not present in the dataset.  
`expected_overall_decision` defaults to `'Pass'` if not labelled.

In [5]:
DIFF_MAP = {
    'A1': 'Easy',   'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',   'C2': 'Hard',
}


def load_dataset(path: Path) -> list:
    """Load JSON or JSONL file into dspy.Example objects with list[RubricQuestion] input."""
    path = Path(path)
    if path.suffix == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
    else:
        rows = []
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))

    examples = []
    for i, row in enumerate(rows, 1):
        question_id       = row.get('question_id', f'Q{i}')
        target_cefr       = row['target_cefr']
        target_difficulty = row.get('target_difficulty') or DIFF_MAP.get(target_cefr, 'Medium')
        language_variant  = row.get('language_variant', 'British English')

        q = RubricQuestion(
            question_id=question_id,
            stem=row['stem'],
            options=row['options'],
            correct_answer=row['correct_answer'],
            explanation=row['explanation'],
            target_cefr=target_cefr,
            target_difficulty=target_difficulty,
            language_variant=language_variant,
        )

        examples.append(
            dspy.Example(
                question_id=question_id,
                questions=[q],
                target_cefr=target_cefr,
                target_difficulty=target_difficulty,
                expected_overall_decision=row.get('expected_overall_decision', 'Pass'),
            ).with_inputs('questions')
        )
    return examples


trainset = load_dataset(DATA_DIR / 'training_dataset_standard.json')
evalset  = load_dataset(DATA_DIR / 'eval_dataset_standard.json')

print(f'Training dataset : {len(trainset)} examples')
print(f'Eval dataset     : {len(evalset)} examples')
print()

cefr_dist    = Counter(ex.target_cefr for ex in trainset)
cefr_dist_ev = Counter(ex.target_cefr for ex in evalset)
print(f'Training CEFR distribution: {dict(sorted(cefr_dist.items()))}')
print(f'Eval     CEFR distribution: {dict(sorted(cefr_dist_ev.items()))}')
print()

ex = trainset[0]
q0 = ex.questions[0]
print(f'Sample — question_id     : {q0.question_id}')
print(f'         target_cefr     : {q0.target_cefr}')
print(f'         language_variant: {q0.language_variant}')
print(f'         stem            : {q0.stem[:80]}...')
print(f'         options         : {q0.options}')

Training dataset : 24 examples
Eval dataset     : 24 examples

Training CEFR distribution: {'A1': 4, 'A2': 4, 'B1': 4, 'B2': 4, 'C1': 4, 'C2': 4}
Eval     CEFR distribution: {'A1': 4, 'A2': 4, 'B1': 4, 'B2': 4, 'C1': 4, 'C2': 4}

Sample — question_id     : Q1
         target_cefr     : A1
         language_variant: British English
         stem            : Read the sentence carefully and answer the question.

The boy is carrying a red ...
         options         : ['A red backpack', 'A blue suitcase', 'A lunch box', 'A football']


## Cell 6 — Baseline Evaluation (Before Optimization)

In [6]:
def evaluate_agent(agent, dataset, label=''):
    """Run agent on every example. Reads pred.output.results[0] (RubricResult)."""
    records = []
    for ex in dataset:
        try:
            pred   = agent(questions=ex.questions)
            score  = rubric_metric(ex, pred)
            result = pred.output.results[0] if pred.output and pred.output.results else None
        except Exception as e:
            pred, score, result = None, 0.0, None
            print(f'  [ERROR] {ex.question_id}: {e}')

        records.append({
            'question_id': ex.question_id,
            'gold':        ex,
            'result':      result,
            'score':       score,
        })

    avg = sum(r['score'] for r in records) / len(records) if records else 0.0
    print(f'[{label}]  N={len(records)}  avg_score={avg:.3f}')
    return records, avg


baseline_agent = RubricJudgeAgent()
print('Running baseline evaluation on training dataset...')
baseline_records, baseline_score = evaluate_agent(baseline_agent, trainset, label='Baseline')

print()
print(f'{"Q-ID":<6} {"target":<8} {"expected":<8} {"decision":<10} {"ambiguity":<16} {"score":>5}')
print('-' * 60)
for r in baseline_records:
    gold = r['gold']
    res  = r['result']
    print(f"{r['question_id']:<6} {gold.target_cefr:<8} {gold.expected_overall_decision:<8} "
          f"{(res.overall_decision if res else 'ERR'):<10} "
          f"{(res.ambiguity if res else 'ERR'):<16} "
          f"{r['score']:>5.1f}")
print('-' * 60)
print(f'Baseline avg : {baseline_score:.3f}')

Running baseline evaluation on training dataset...
[Baseline]  N=24  avg_score=0.917

Q-ID   target   expected decision   ambiguity        score
------------------------------------------------------------
Q1     A1       Pass     Pass       No Issue           1.0
Q2     A2       Pass     Pass       No Issue           1.0
Q3     B1       Pass     Revise     No Issue           0.5
Q4     B2       Pass     Pass       No Issue           1.0
Q5     C1       Pass     Revise     No Issue           0.5
Q6     C2       Pass     Pass       No Issue           1.0
Q7     A1       Pass     Pass       No Issue           1.0
Q8     A2       Pass     Pass       No Issue           1.0
Q9     B1       Pass     Pass       No Issue           1.0
Q10    B2       Pass     Pass       No Issue           1.0
Q11    C1       Pass     Pass       No Issue           1.0
Q12    C2       Pass     Pass       No Issue           1.0
Q13    A1       Pass     Pass       No Issue           1.0
Q14    A2       Pass     Pa

## Cell 7 — Run Optimizer

Tries **GEPA** first — rewrites the rubric prompt to improve `overall_decision` accuracy.  
Falls back to **BootstrapFewShot** if GEPA raises any error.  
Saves to `artifacts/rubric_agent_optimized.json`.

In [7]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=PROJECT_ROOT / '.env')


def run_gepa(trainset):
    from dspy.teleprompt import GEPA

    reflection_lm = dspy.LM(
        f"openai/{os.getenv('MISTRAL_MODEL', 'open-mistral-nemo')}",
        api_key=os.environ['MISTRAL_API_KEY'],
        api_base=os.getenv('MISTRAL_API_BASE', 'https://api.mistral.ai/v1'),
        temperature=0.9,
        max_tokens=4000,
    )

    log_dir = ARTIFACTS_DIR / 'gepa_rubric_logs'
    log_dir.mkdir(exist_ok=True)

    optimizer = GEPA(
        metric=rubric_metric,
        auto='light',
        reflection_lm=reflection_lm,
        reflection_minibatch_size=3,
        log_dir=str(log_dir),
        track_stats=True,
        seed=42,
    )

    student = RubricJudgeAgent()
    split   = max(2, int(len(trainset) * 0.8))
    print(f'  GEPA: {split} train / {len(trainset) - split} val examples')

    optimized = optimizer.compile(
        student,
        trainset=trainset[:split],
        valset=trainset[split:],
    )
    return optimized, 'GEPA'


def run_bootstrap(trainset):
    from dspy.teleprompt import BootstrapFewShot
    optimizer = BootstrapFewShot(
        metric=rubric_metric,
        max_bootstrapped_demos=3,
        max_labeled_demos=4,
    )
    student = RubricJudgeAgent()
    print(f'  BootstrapFewShot: {len(trainset)} examples')
    optimized = optimizer.compile(student, trainset=trainset)
    return optimized, 'BootstrapFewShot'


print('Attempting GEPA optimization...')
try:
    optimized_agent, optimizer_used = run_gepa(trainset)
    print('GEPA complete.')
except Exception as gepa_err:
    print(f'GEPA failed: {gepa_err}')
    print('Falling back to BootstrapFewShot...')
    optimized_agent, optimizer_used = run_bootstrap(trainset)
    print('BootstrapFewShot complete.')

optimized_agent.save(str(OPTIMIZED_PATH))
print(f'\nOptimizer used : {optimizer_used}')
print(f'Artifact saved : {OPTIMIZED_PATH}')

2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 400 metric calls of the program. This amounts to 16.67 full evals on the train+val set.
2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


Attempting GEPA optimization...
  GEPA: 19 train / 5 val examples


GEPA Optimization:   0%|          | 0/400 [00:00<?, ?rollouts/s]2026/04/07 09:22:58 INFO dspy.evaluate.evaluate: Average Metric: 4.0 / 5 (80.0%)
2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.8 over 5 / 5 examples
GEPA Optimization:   1%|▏         | 5/400 [00:00<00:14, 26.53rollouts/s]2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.8


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 32.01it/s]

2026/04/07 09:22:58 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   2%|▏         | 8/400 [00:00<00:15, 25.46rollouts/s]2026/04/07 09:22:58 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.8


...

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 19.14it/s]

2026/04/07 09:24:58 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/07 09:24:58 INFO dspy.teleprompt.gepa.gepa: Iteration 124: All subsample scores perfect. Skipping.
2026/04/07 09:24:58 INFO dspy.teleprompt.gepa.gepa: Iteration 124: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 399/400 [01:59<00:00, 15.78rollouts/s]2026/04/07 09:24:58 INFO dspy.teleprompt.gepa.gepa: Iteration 125: Selected program 2 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 18.84it/s]

2026/04/07 09:24:58 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/07 09:24:58 INFO dspy.teleprompt.gepa.gepa: Iteration 125: All subsample scores perfect. Skipping.
2026/04/07 09:24:58 INFO dspy.teleprompt.gepa.gepa: Iteration 125: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 399/400 [02:00<00:00,  3.32rollouts/s]


GEPA complete.

Optimizer used : GEPA
Artifact saved : D:\Topin\artifacts\rubric_agent_optimized.json


## Cell 8 — Load Optimized Agent & Post-Optimization Evaluation

In [8]:
loaded_agent = RubricJudgeAgent()
loaded_agent.load(str(OPTIMIZED_PATH))
print(f'Loaded optimized agent from {OPTIMIZED_PATH}')

print('\nRunning post-optimization evaluation on training dataset...')
optimized_records, optimized_score = evaluate_agent(loaded_agent, trainset, label='Optimized')

print()
print(f'{"Q-ID":<6} {"target":<8} {"expected":<8} {"decision":<10} {"ambiguity":<16} {"score":>5}')
print('-' * 60)
for r in optimized_records:
    gold = r['gold']
    res  = r['result']
    print(f"{r['question_id']:<6} {gold.target_cefr:<8} {gold.expected_overall_decision:<8} "
          f"{(res.overall_decision if res else 'ERR'):<10} "
          f"{(res.ambiguity if res else 'ERR'):<16} "
          f"{r['score']:>5.1f}")
print('-' * 60)
print(f'Baseline avg  : {baseline_score:.3f}')
print(f'Optimized avg : {optimized_score:.3f}')
print(f'Delta         : {optimized_score - baseline_score:+.3f}')

Loaded optimized agent from D:\Topin\artifacts\rubric_agent_optimized.json

Running post-optimization evaluation on training dataset...
[Optimized]  N=24  avg_score=1.000

Q-ID   target   expected decision   ambiguity        score
------------------------------------------------------------
Q1     A1       Pass     Pass       No Issue           1.0
Q2     A2       Pass     Pass       No Issue           1.0
Q3     B1       Pass     Pass       No Issue           1.0
Q4     B2       Pass     Pass       No Issue           1.0
Q5     C1       Pass     Pass       No Issue           1.0
Q6     C2       Pass     Pass       No Issue           1.0
Q7     A1       Pass     Pass       No Issue           1.0
Q8     A2       Pass     Pass       No Issue           1.0
Q9     B1       Pass     Pass       No Issue           1.0
Q10    B2       Pass     Pass       No Issue           1.0
Q11    C1       Pass     Pass       No Issue           1.0
Q12    C2       Pass     Pass       No Issue           1.0


## Cell 9 — Write Promptfoo Provider

Standalone Python file imported by Promptfoo.  
Accepts a JSON array of question objects (with `language_variant`), runs the optimized rubric judge.

In [9]:
provider_code = '''\
from __future__ import annotations
import json, os, sys
from pathlib import Path
from typing import Literal
from pydantic import BaseModel

EVALS_DIR    = Path(__file__).resolve().parent
PROJECT_ROOT = EVALS_DIR.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_venv_sp = PROJECT_ROOT / \'.venv\' / \'Lib\' / \'site-packages\'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / \'.venv\' / \'lib\' / \'site-packages\'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))

from utils import configure_dspy_from_env
import dspy


class RubricQuestion(BaseModel):
    question_id:       str
    stem:              str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str
    language_variant:  str


class RubricResult(BaseModel):
    question_id:                          str
    grammatical_accuracy:                 Literal[\'No Issues\', \'Minor Issues\', \'Major Issues\']
    spelling:                             Literal[\'No Issues\', \'Minor Issues\', \'Major Issues\']
    ambiguity:                            Literal[\'No Issue\', \'Minor Issue\', \'Major Issue\']
    functionality_alignment:              Literal[\'Aligned\', \'Partially Aligned\', \'Not Aligned\']
    instruction_clarity_appropriateness:  Literal[\'Clear\', \'Needs Improvement\', \'Unclear\']
    academic_language_exam_acceptability: Literal[\'Acceptable\', \'Needs Improvement\', \'Not Acceptable\']
    option_explanation_consistency:       Literal[\'Consistent\', \'Inconsistent\']
    readability:                          Literal[\'Good\', \'Needs Improvement\', \'Poor\']
    formatting_spacing:                   Literal[\'No Issues\', \'Minor Issues\', \'Major Issues\']
    punctuation:                          Literal[\'No Issues\', \'Minor Issues\', \'Major Issues\']
    british_american_english_consistency: Literal[\'Consistent\', \'Inconsistent\']
    overall_decision:                     Literal[\'Pass\', \'Revise\', \'Fail\']
    priority_reason:                      str
    revision_feedback:                    str


class RubricOutput(BaseModel):
    results: list[RubricResult]


class RubricJudgeSignature(dspy.Signature):
    """Evaluate a list of MCQ questions using the rubric.
    ambiguity is highest priority — Major Issue forces Fail.
    Return one RubricResult per question in the same order.
    """
    questions: list[RubricQuestion] = dspy.InputField(desc=\'List of MCQ questions to be evaluated\')
    output:    RubricOutput         = dspy.OutputField(desc=\'Rubric evaluation results for all questions\')


class RubricJudgeAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(RubricJudgeSignature)

    def forward(self, questions):
        return self.judge(questions=questions)


_OPTIMIZED = PROJECT_ROOT / "artifacts" / "rubric_agent_optimized.json"
_agent = None


def call_api(prompt: str, options, context):
    global _agent
    configure_dspy_from_env()

    if _agent is None:
        _agent = RubricJudgeAgent()
        if _OPTIMIZED.exists():
            _agent.load(str(_OPTIMIZED))

    try:
        data = json.loads(prompt)
        if isinstance(data, dict):
            data = [data]
        questions = [RubricQuestion(**q) for q in data]
    except Exception as e:
        return {"error": f"Invalid input: {e}"}

    try:
        pred    = _agent(questions=questions)
        results = [r.model_dump() for r in pred.output.results]
        return {"output": json.dumps(results)}
    except Exception as e:
        return {"error": str(e)}
'''

provider_path = EVALS_DIR / 'rubric_eval_provider.py'
provider_path.write_text(provider_code, encoding='utf-8')
print(f'Written: {provider_path}')

Written: D:\Topin\notebooks\evals\rubric_eval_provider.py


## Cell 10 — Build Promptfoo Test Cases

Source: `eval_dataset_standard.json` (24 questions).  
Each test asserts `overall_decision` matches `expected_overall_decision` (default `Pass`).

In [10]:
def build_tests(examples: list) -> list:
    tests = []
    for ex in examples:
        expected_decision = ex.expected_overall_decision
        q = ex.questions[0]
        question_json = json.dumps([q.model_dump()])

        tests.append({
            'description': f'{ex.question_id} | cefr={ex.target_cefr} expect={expected_decision}',
            'vars': {'question_json': question_json},
            'assert': [
                {
                    'type': 'javascript',
                    'value': (
                        f'const results = JSON.parse(output); '
                        f'const p = Array.isArray(results) ? results[0] : results; '
                        f'return p.overall_decision === "{expected_decision}";'
                    ),
                },
            ],
        })
    return tests


all_tests = build_tests(evalset)

print(f'Total eval test cases : {len(all_tests)}  (from eval_dataset_standard.json)')
cefr_dist_tests = Counter(ex.target_cefr for ex in evalset)
print(f'CEFR distribution     : {dict(sorted(cefr_dist_tests.items()))}')
print()
for t in all_tests:
    print(f'  {t["description"]}')

Total eval test cases : 24  (from eval_dataset_standard.json)
CEFR distribution     : {'A1': 4, 'A2': 4, 'B1': 4, 'B2': 4, 'C1': 4, 'C2': 4}

  Q1 | cefr=A1 expect=Pass
  Q2 | cefr=A2 expect=Pass
  Q3 | cefr=B1 expect=Pass
  Q4 | cefr=B2 expect=Pass
  Q5 | cefr=C1 expect=Pass
  Q6 | cefr=C2 expect=Pass
  Q7 | cefr=A1 expect=Pass
  Q8 | cefr=A2 expect=Pass
  Q9 | cefr=B1 expect=Pass
  Q10 | cefr=B2 expect=Pass
  Q11 | cefr=C1 expect=Pass
  Q12 | cefr=C2 expect=Pass
  Q13 | cefr=A1 expect=Pass
  Q14 | cefr=A2 expect=Pass
  Q15 | cefr=B1 expect=Pass
  Q16 | cefr=B2 expect=Pass
  Q17 | cefr=C1 expect=Pass
  Q18 | cefr=C2 expect=Pass
  Q19 | cefr=A1 expect=Pass
  Q20 | cefr=A2 expect=Pass
  Q21 | cefr=B1 expect=Pass
  Q22 | cefr=B2 expect=Pass
  Q23 | cefr=C1 expect=Pass
  Q24 | cefr=C2 expect=Pass


## Cell 11 — Write Promptfoo Config YAML

In [11]:
import yaml

config = {
    'description': 'RubricJudge eval — overall_decision per question',
    'providers': [{'id': 'file://rubric_eval_provider.py'}],
    'prompts': ['{{question_json}}'],
    'tests': all_tests,
}

config_path = EVALS_DIR / 'rubric_eval_config.yaml'
with config_path.open('w', encoding='utf-8') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'Written: {config_path}')

Written: D:\Topin\notebooks\evals\rubric_eval_config.yaml


## Cell 12 — Run Promptfoo Eval as Subprocess

In [12]:
cmd = [
    'npx', 'promptfoo@latest', 'eval',
    '-c', str(config_path),
    '-o', str(EVAL_OUTPUT),
    '--output-format', 'json',
    '--no-cache',
    '--max-concurrency', '1',
]

print('Running Promptfoo eval...')
print(f'  Config : {config_path}')
print(f'  Output : {EVAL_OUTPUT}')
print()

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    shell=True,
    cwd=str(EVALS_DIR),
    timeout=600,
)

print('--- STDOUT (last 3000 chars) ---')
stdout = result.stdout
print(stdout[-3000:] if len(stdout) > 3000 else stdout)

if result.returncode != 0:
    print('--- STDERR ---')
    stderr = result.stderr
    print(stderr[-2000:] if len(stderr) > 2000 else stderr)
    print(f'\nPromptfoo exited with code {result.returncode}')
else:
    print('\nPromptfoo completed successfully (exit code 0).')

Running Promptfoo eval...
  Config : D:\Topin\notebooks\evals\rubric_eval_config.yaml
  Output : D:\Topin\notebooks\evals\rubric_eval_output.json

--- STDOUT (last 3000 chars) ---

--- STDERR ---
ests from
  --filter-metadata <key=value>                     Only run tests whose metadata matches the key=value pair. Can be specified multiple times for AND logic (e.g. --filter-metadata type=unit --filter-metadata env=prod)
  -o, --output <paths...>                           Path to output file (csv, txt, json, yaml, yml, html), default is no output file
  --table                                           Output table in CLI (default: true)
  --no-table                                        Do not output table in CLI
  --table-cell-max-length <number>                  Truncate console table cells to this length (default: "250")
  --share                                           Create a shareable URL
  --no-share                                        Do not share, this overrides the con

## Cell 13 — Parse & Display Promptfoo Results

In [14]:
if not EVAL_OUTPUT.exists():
    print(f'Output file not found: {EVAL_OUTPUT}')
    print('Re-run the Promptfoo cell above.')
else:
    raw_text = EVAL_OUTPUT.read_text(encoding='utf-8').strip()
    if not raw_text:
        print(f'Output file is empty: {EVAL_OUTPUT}')
        print('Check the STDOUT / STDERR printed in the cell above.')
    else:
        try:
            eval_data = json.loads(raw_text)
        except json.JSONDecodeError as e:
            print(f'Could not parse output file: {e}')
            print('First 500 chars:')
            print(raw_text[:500])
        else:
            raw          = eval_data.get('results', {})
            results_list = raw.get('results', raw) if isinstance(raw, dict) else raw
            stats        = raw.get('stats', {}) if isinstance(raw, dict) else {}

            passed = sum(1 for r in results_list if r.get('success', False))
            total  = len(results_list)

            print(f'Promptfoo Eval Results')
            print(f'  Passed : {passed}/{total}')
            print(f'  Failed : {total - passed}/{total}')
            if stats:
                print(f'  Stats  : {stats}')
            print()

            print(f'{"Q-ID":<10} {"Description":<40} {"Result":<7} {"decision":<10} {"ambiguity"}')
            print('-' * 85)

            for r in results_list:
                desc    = r.get('description', '')[:39]
                success = 'PASS' if r.get('success') else 'FAIL'
                raw_out = r.get('response', {}).get('output', '[]')
                try:
                    p_list   = json.loads(raw_out)
                    p        = p_list[0] if isinstance(p_list, list) and p_list else p_list
                    qid      = p.get('question_id', 'N/A')
                    decision = p.get('overall_decision', 'N/A')
                    ambiguity = p.get('ambiguity', 'N/A')
                except Exception:
                    qid = decision = ambiguity = 'PARSE_ERR'

                print(f'{qid:<10} {desc:<40} {success:<7} {decision:<10} {ambiguity}')

            print()
            print(f'Full eval output saved to: {EVAL_OUTPUT}')

Output file is empty: D:\Topin\notebooks\evals\rubric_eval_output.json
Check the STDOUT / STDERR printed in the cell above.
